# Simulation Creation

In [1]:
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql.types as T
import pyspark.sql as sql
import pyspark.sql.functions as F
import numpy as np
import datetime as dt
from random import randint
import subprocess
from time import time

os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.security.manager=allow"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.extraJavaOptions=-Djava.security.manager=allow pyspark-shell"

#Mandatory
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

In [2]:
START_DATE : str = "2025-01-01"						# Simulation start date
START_TIME : str = "06:00:00"						# Simulation start time
NB_DAYS : int = 5									# Number of days to simulate
EDGES_MAX_SPEED : float = 27.78						# Max speed of the edges in m/s (120 km/h)
TRAIN_SPEED : float = 33.33							# Train speed in m/s (120 km/h)
DIRECTORY : str = "sumo_data"
OUTPUT_DIRECTORY : str = "new_start"
START_DATETIME : dt.datetime = dt.datetime.strptime(f"{START_DATE} {START_TIME}", "%Y-%m-%d %H:%M:%S")
END_DATETIME : dt.datetime = START_DATETIME + dt.timedelta(days=NB_DAYS)

filenames = {
	"switches" : f"{DIRECTORY}/switches.csv",
	"tracks" : f"{DIRECTORY}/main_tracks.csv",
	# "platforms" : f"{DIRECTORY}/station_track_assigned.csv",
	"platforms" : f"station_track_assigned.csv",
	"punctuality" : f"{DIRECTORY}/punctuality/202501.csv",
	"trains" : f"{DIRECTORY}/trains.add.xml"
}

output = {
	"switches" : f"{OUTPUT_DIRECTORY}/switches.nod.xml",
	"tracks" : f"{OUTPUT_DIRECTORY}/tracks.edg.xml",
	"routes" : f"{OUTPUT_DIRECTORY}/routes.rou.xml",
	"network" : f"{OUTPUT_DIRECTORY}/network.net.xml",
	"platforms" : f"{OUTPUT_DIRECTORY}/platforms.add.xml",
	"schedule" : f"{OUTPUT_DIRECTORY}/schedule.trips.xml",
	"config" : f"{OUTPUT_DIRECTORY}/config.sumocfg",
	"trains" : f"{OUTPUT_DIRECTORY}/trains.add.xml",
	"weight_src" : f"{OUTPUT_DIRECTORY}/weights.src.xml",
	"weight_dst" : f"{OUTPUT_DIRECTORY}/weights.dst.xml",
}

In [3]:
spark : sql.SparkSession = (sql.SparkSession.builder
	.appName("RailwaySimulationGenerator")
	.config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
	.getOrCreate()
)

## Data extraction

### Switches

In [4]:
switches_df = spark.read.csv(filenames["switches"], header=True, inferSchema=True, sep=";")
swtiches_dict = dict()
for row in switches_df.collect() :
	switch_id : int = int(row["ID"])
	if switch_id not in swtiches_dict :
		swtiches_dict[switch_id] = {}
	else :
		print(f"Two stations have the same ID {switch_id}")
	swtiches_dict[switch_id]["x"] = row["Y"]
	swtiches_dict[switch_id]["y"] = row["X"]

In [5]:
switches_df.show(5, truncate=False)

+---+---------+--------+
|ID |X        |Y       |
+---+---------+--------+
|0.0|50.816795|4.395716|
|1.0|50.811235|4.399232|
|2.0|50.811562|4.399146|
|3.0|50.810215|4.398703|
|4.0|50.810734|4.399025|
+---+---------+--------+
only showing top 5 rows


In [6]:
for i in range(5) :
	switch_id = randint(0, len(swtiches_dict))
	print(switch_id, swtiches_dict[switch_id])

17175 {'x': 4.486953, 'y': 51.020587}
1791 {'x': 4.338935, 'y': 50.773965}
222 {'x': 5.658096, 'y': 50.881303}
20643 {'x': 3.551887, 'y': 51.069695}
4457 {'x': 4.337087, 'y': 50.837102}


### Tracks

In [5]:
tracks_df = spark.read.csv(filenames["tracks"], header=True, inferSchema=True, sep=";")
tracks_dict = dict()
schema : T.DataType = T.ArrayType(T.ArrayType(T.DoubleType()))
tracks_df = tracks_df.withColumn(
	"Path",
	F.from_json(F.col("Path"), schema)
)
for row in tracks_df.collect() :
	track_id = int(row["ID"])
	start = row["Departure_switch"]
	end = row["Arrival_switch"]
	shape = [(s[1], s[0]) for s in row["Path"]]
	distance = round(row["Length_m"], 6) 
	if track_id not in tracks_dict :
		tracks_dict[track_id] = {}
	tracks_dict[track_id] = {
		"start" : start,
		"end" : end,
		"shape" : shape,
		"distance" : distance
	}

In [8]:
tracks_df.show(5, truncate=False)

+---+----------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
for i in range(5) :
	track_id = randint(0, len(tracks_dict))
	print(track_id, tracks_dict[track_id])

24542 {'start': 11168, 'end': 1539, 'shape': [(5.575209, 50.607883), (5.575154, 50.607822), (5.575111, 50.60777), (5.575071, 50.607718), (5.575026, 50.607653), (5.574991, 50.607599), (5.574959, 50.607544), (5.57487, 50.607382), (5.574844, 50.607338), (5.574816, 50.607294), (5.574779, 50.607242), (5.574752, 50.607208), (5.574724, 50.607175), (5.574671, 50.607118), (5.57464, 50.607086), (5.574597, 50.607044), (5.574564, 50.607012), (5.574518, 50.606971), (5.574471, 50.60693), (5.574423, 50.606891), (5.574386, 50.606861), (5.574309, 50.606804), (5.574256, 50.606766), (5.574202, 50.606729), (5.574146, 50.606693), (5.574075, 50.606649), (5.574017, 50.606615), (5.573943, 50.606574), (5.573882, 50.606542), (5.57382, 50.60651), (5.573757, 50.60648), (5.573709, 50.606457), (5.573498, 50.606364)], 'distance': 212.506381}
23712 {'start': 16057, 'end': 20178, 'shape': [(5.692692, 50.739019), (5.69269, 50.738965), (5.692688, 50.738878), (5.692665, 50.738645)], 'distance': 41.656359}
2647 {'start': 

### Station platforms

In [6]:
plaforms_df = spark.read.csv(filenames["platforms"], header=True, inferSchema=True, sep=",")
platforms_dict = dict()
for row in plaforms_df.collect() :
	station_id = int(row["Station_ID"])
	track_id = row["track_id"]
	city = row["Station_name"]
	if station_id not in platforms_dict :
		platforms_dict[station_id] = {
			"name" : city,
			"platforms" : []
		}
	platforms_dict[station_id]["platforms"].append(track_id)

In [11]:
plaforms_df.show(5, truncate=False)

+----------+------------+-----------+---------------+--------+-----------------+------------------+----------------------+-----------------+-----------------+---------------+---------------+---------------+
|Station_ID|Station_name|n_platforms|search_radius_m|track_id|distance_m       |track_length_m    |position_along_track_m|proj_x_m         |proj_y_m         |rank_in_station|candidate_class|candidate_color|
+----------+------------+-----------+---------------+--------+-----------------+------------------+----------------------+-----------------+-----------------+---------------+---------------+---------------+
|6.0       |AALST       |7          |280            |26563   |5.472611916381333|507.35274826470726|294.9386336072009     |626797.5262348288|681470.3624483546|1              |best           |green          |
|6.0       |AALST       |7          |280            |26562   |5.776529336635288|562.6095217545319 |211.82013545637201    |626804.1904653596|681479.4250651479|2             

In [12]:
i = 0
for station_id in platforms_dict :
	print(station_id, platforms_dict[station_id])
	i += 1
	if i >= 5 :
		break

6 {'name': 'AALST', 'platforms': [26563, 26562, 3559, 26623, 23191, 26565, 26564]}
8 {'name': 'AALTER', 'platforms': [7633, 7632, 7772, 24311]}
9 {'name': 'AARSCHOT', 'platforms': [13001, 12601, 13007, 12513, 12602]}
10 {'name': 'AARSELE', 'platforms': [9664, 9663]}
12 {'name': 'ACREN', 'platforms': [1992, 1991]}


### Punctuality Data

In [7]:
punctuality_data_df = spark.read.csv(filenames["punctuality"], header=True, inferSchema=True, sep=";")
window : sql.Window = (
	sql.Window.partitionBy("TRAIN_NO","REAL_DATE_DEP") 
	.orderBy("PLANNED_DATETIME_DEP")
)
punctuality_data_df = (
	punctuality_data_df
	.withColumn(
		"NEXT_STOPPING_PLACE_ID",
		F.lead("STOPPING_PLACE_ID").over(window)
	)
)
punctuality_data_df = (punctuality_data_df.filter(
	(F.col("PLANNED_DATETIME_DEP") >= F.lit(START_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))) & 
	(F.col("PLANNED_DATETIME_DEP") <= F.lit(END_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))))
	.orderBy("TRAIN_NO", "REAL_DATE_DEP")
)

In [14]:
punctuality_data_df.show(5, truncate=False)

+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|RELATION_DIRECTION                      |LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|10      |ICE     |SNCB/NMBS |825              |37         |243      |243      |ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID|37         |01-01-2025   |01-01-2025   |2025-01-01 20:28:00 |2025-01-01 20:28:00 |266                   |
|10      |ICE     |SNCB/NMBS |266              |37         |297      |297      |ICE: FRANKFU

## Network Creation

### Switches file

In [8]:
start_time = time()
switches_str = '<?xml version="1.0" encoding="UTF-8"?>\n' + '<nodes>\n'
for switch_id in swtiches_dict :
	switch = swtiches_dict[switch_id]
	switches_str += f'\t<node id="{switch_id}" x="{switch["x"]}" y="{switch["y"]}" type="priority"/>\n'
switches_str += '</nodes>'
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")
# print(switches_str)

Total execution time: 0.08 seconds.


In [9]:
with open(output["switches"], 'w', encoding = "utf-8") as f :
		f.write(switches_str)

### Tracks file

In [10]:
start_time = time()
tracks_str = '<?xml version="1.0" encoding="UTF-8"?>\n' + '<edges>\n'
cmpt = 0
platforms_ids = dict()
edges_ids = dict()
edges = dict()
for track_id in tracks_dict :
	track = tracks_dict[track_id]
	start = track["start"]
	end = track["end"]
	edges_ids[cmpt] = track_id
	edges[cmpt] = (start, end)
	tracks_str += f'\t<edge id="{cmpt}" from="{start}" to="{end}" priority="2" numLanes="1" length="{track["distance"]}" speed="{EDGES_MAX_SPEED}" allow="rail" shape="'
	# tracks_str += f'\t<edge id="{track_id}" from="{start}" to="{end}" priority="2" numLanes="1" length="{track["distance"]}" speed="{EDGES_MAX_SPEED}" allow="rail"/>\n'
	for coords in track["shape"] :
		tracks_str += f'{coords[0]},{coords[1]} '
	tracks_str += f'"/>\n'
	cmpt += 1
	edges_ids[cmpt] = track_id
	edges[cmpt] = (end, start)
	tracks_str += f'\t<edge id="{cmpt}" from="{end}" to="{start}" priority="2" numLanes="1" length="{track["distance"]}" speed="{EDGES_MAX_SPEED}" allow="rail" shape="'
	for coords in track["shape"][::-1] :
		tracks_str += f'{coords[0]},{coords[1]} '
	tracks_str += f'"/>\n'
	ids = (cmpt - 1, cmpt)
	for station in platforms_dict :
		if track_id in platforms_dict[station]["platforms"] :
			if station not in platforms_ids :
				platforms_ids[station] = set()
			platforms_ids[station].add(ids[0])
			platforms_ids[station].add(ids[1])
			break
	cmpt += 1
tracks_str += '</edges>'
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")
# print(tracks_str)

Total execution time: 129.53 seconds.


In [11]:
with open(output["tracks"], 'w', encoding = "utf-8") as f :
		f.write(tracks_str)

### Platforms file

In [12]:
start_time = time()
platforms_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<additional>\n'
)
ptaform_id = 0
for station_id in platforms_ids :
	city = platforms_dict[station_id]["name"]
	for i, track_id in enumerate(platforms_ids[station_id]) :
	# for i, track_id in enumerate(station) :
		platforms_str += f'\t<trainStop id="{ptaform_id}" lane="{track_id}_0" name="{city} platform {i + 1}" startPos="{80 if tracks_dict[edges_ids[track_id]]["distance"] > 250 else 10}" endPos="{ 200 if tracks_dict[edges_ids[track_id]]["distance"] >= 250 else tracks_dict[edges_ids[track_id]]["distance"] - 30}" />\n'
		ptaform_id += 1
platforms_str += '</additional>'
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")
# print(platforms_str)

Total execution time: 0.02 seconds.


In [13]:
with open(output["platforms"], 'w', encoding = "utf-8") as f :
		f.write(platforms_str)

### Network generation

In [14]:
network_command = [
	"netconvert",					
	"--node-files", f'{output["switches"]}',	
	"--edge-files", f'{output["tracks"]}',	
	"--railway.signal.guess.by-stops", "true",			
	"--output-file", f'{output["network"]}',		
	"--proj.utm", "true"		
]

In [15]:
# print(" ".join(network_command))
start_time = time()
subprocess.run(network_command, check=True)
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")

Total execution time: 30.76 seconds.


## Trip creation

### Weight station

In [16]:
start_time = time()
weight_stations_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<edgedata>\n' +
	f'\t<interval begin="0" end="{1 * 24 * 3600}">\n'
)

for station_id in platforms_ids :
	for track_id in platforms_ids[station_id] :
		weight_stations_str += f'\t\t<edge id="{track_id}" value="1">\n'

weight_stations_str += (
	'\t</interval>\n' + 
	'</edgedata>'
)
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")
# print(weight_stations_str)

Total execution time: 0.01 seconds.


In [17]:
with open(output["weight_src"], 'w', encoding = "utf-8") as f :
	f.write(weight_stations_str)

In [18]:
with open(output["weight_dst"], 'w', encoding = "utf-8") as f :
	f.write(weight_stations_str)

### Reconstruct order of stations

In [23]:
punctuality_data_df = punctuality_data_df.toPandas()

trips = {}

for train_id, group in punctuality_data_df.groupby("RELATION"):
	stops = group["STOPPING_PLACE_ID"].tolist()
	# next_stops = group["NEXT_STOPPING_PLACE_ID"].tolist()

	path = []
	for s in stops :
		if s in path :
			break
		path.append(s) 
	trips[train_id] = path

In [24]:
punctuality_data_df.groupby("TRAIN_NO").head(20)

,TRAIN_NO,RELATION,TRAIN_SERV,STOPPING_PLACE_ID,LINE_NO_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,LINE_NO_ARR,REAL_DATE_ARR,REAL_DATE_DEP,PLANNED_DATETIME_ARR,PLANNED_DATETIME_DEP,NEXT_STOPPING_PLACE_ID
0,10,ICE,SNCB/NMBS,825,37,243.0,243,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:28:00,2025-01-01 20:28:00,266.0
1,10,ICE,SNCB/NMBS,266,37,297.0,297,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,3,01-01-2025,01-01-2025,2025-01-01 20:39:00,2025-01-01 20:39:00,27.0
2,10,ICE,SNCB/NMBS,27,37,276.0,276,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:40:00,2025-01-01 20:40:00,726.0
3,10,ICE,SNCB/NMBS,726,36,223.0,127,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:43:00,2025-01-01 20:46:00,31.0
4,10,ICE,SNCB/NMBS,31,2,62.0,62,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,36,01-01-2025,01-01-2025,2025-01-01 20:51:00,2025-01-01 20:51:00,715.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267076,19976,IC 19-2,SNCB/NMBS,1154,94,NaN,3,IC 19-2: TOURNAI -> LILLE FLANDRES,None,None,04-01-2025,NaT,2025-01-04 21:44:00,427.0
267077,19976,IC 19-2,SNCB/NMBS,427,94,-32.0,-25,IC 19-2: TOURNAI -> LILLE FLANDRES,94,04-01-2025,04-01-2025,2025-01-04 21:48:00,2025-01-04 21:49:00,NaN
267078,19976,IC 19-2,SNCB/NMBS,1154,94,NaN,11,IC 19-2: TOURNAI -> LILLE FLANDRES,None,None,05-01-2025,NaT,2025-01-05 21:44:00,427.0
267079,19976,IC 19-2,SNCB/NMBS,427,94,-22.0,-15,IC 19-2: TOURNAI -> LILLE FLANDRES,94,05-01-2025,05-01-2025,2025-01-05 21:48:00,2025-01-05 21:49:00,NaN


In [24]:
for train_id in trips :
	path = []
	for s in trips[train_id] :
		path.append(platforms_dict[s]["name"]) if s in platforms_dict else None
	print(train_id, path)

EURST ['HALLE', 'FOREST-MIDI', 'BRUXELLES-MIDI', 'BRUXELLES-CHAPELLE', 'BRUXELLES-CENTRAL', 'BRUXELLES-CONGRES', 'BRUXELLES-NORD', 'SCHAERBEEK', 'MECHELEN', 'MECHELEN-NEKKERSPOEL', 'SINT-KATELIJNE-WAVER', 'DUFFEL', 'KONTICH-LINT', 'HOVE', 'MORTSEL-OUDE GOD', 'MORTSEL', 'ANTWERPEN-BERCHEM', 'ANTWERPEN-CENTRAAL', 'ANTWERPEN-LUCHTBAL', 'NOORDERKEMPEN']
EXTRA ['PUURS', 'RUISBROEK-SAUVEGARDE', 'BOOM', 'NIEL', 'SCHELLE', 'HEMIKSEM', 'HOBOKEN-POLDER']
IC 01 ['WELKENRAEDT']
IC 02 ['GENT-SINT-PIETERS', 'GENTBRUGGE', 'GENT-DAMPOORT', 'BEERVELDE', 'LOKEREN', 'SINAAI', 'BELSELE', 'SINT-NIKLAAS', 'NIEUWKERKEN-WAAS', 'BEVEREN(WAAS)', 'MELSELE', 'ZWIJNDRECHT', 'ANTWERPEN-ZUID', 'ANTWERPEN-BERCHEM']
IC 03 ['GENT-SINT-PIETERS', 'ANDERLECHT', 'BRUXELLES-MIDI', 'BRUXELLES-CHAPELLE', 'BRUXELLES-CENTRAL', 'BRUXELLES-CONGRES', 'BRUXELLES-NORD', 'SCHAERBEEK', 'HAREN-SUD', 'DIEGEM', 'ZAVENTEM', 'NOSSEGEM', 'KORTENBERG', 'ERPS-KWERPS', 'VELTEM', 'HERENT', 'LEUVEN', 'VERTRIJK', 'TIENEN', 'EZEMAAL', 'NEERWINDEN'

In [25]:
final_trips  = {}
for trip in trips :
	path = []
	for s in trips[trip] :
		path.append(s) if s in platforms_dict else None
	if len(path) > 1 :
		final_trips[trip] = path

In [26]:
for train_id in final_trips :
	path = []
	for s in final_trips[train_id] :
		# path.append(platforms_dict[s]["name"]) if s in platforms_dict else None
		path.append((s, platforms_dict[s]["name"], s in platforms_dict))
	print(train_id, path)

EURST [(504, 'HALLE', True), (415, 'FOREST-MIDI', True), (220, 'BRUXELLES-MIDI', True), (217, 'BRUXELLES-CHAPELLE', True), (215, 'BRUXELLES-CENTRAL', True), (216, 'BRUXELLES-CONGRES', True), (221, 'BRUXELLES-NORD', True), (1048, 'SCHAERBEEK', True), (810, 'MECHELEN', True), (811, 'MECHELEN-NEKKERSPOEL', True), (1083, 'SINT-KATELIJNE-WAVER', True), (336, 'DUFFEL', True), (644, 'KONTICH-LINT', True), (590, 'HOVE', True), (866, 'MORTSEL-OUDE GOD', True), (863, 'MORTSEL', True), (139, 'ANTWERPEN-BERCHEM', True), (37, 'ANTWERPEN-CENTRAAL', True), (764, 'ANTWERPEN-LUCHTBAL', True), (1839, 'NOORDERKEMPEN', True)]
EXTRA [(977, 'PUURS', True), (1017, 'RUISBROEK-SAUVEGARDE', True), (188, 'BOOM', True), (905, 'NIEL', True), (1066, 'SCHELLE', True), (546, 'HEMIKSEM', True), (570, 'HOBOKEN-POLDER', True)]
IC 02 [(455, 'GENT-SINT-PIETERS', True), (447, 'GENTBRUGGE', True), (449, 'GENT-DAMPOORT', True), (130, 'BEERVELDE', True), (748, 'LOKEREN', True), (1073, 'SINAAI', True), (138, 'BELSELE', True), 

In [27]:
temp = {}
for track_id, track in edges.items():
	# start, end = track["start"], track["end"]
	start, end = track[0], track[1]
	if start not in temp :
		temp[start] = set()
	temp[start].add(track_id)
	if end not in temp :
		temp[end] = set()
	temp[end].add(track_id)

In [28]:
import networkx as nx

G = nx.Graph()

for node in temp :
	tracks = list(temp[node])
	for i in range(len(tracks)) :
		for j in range(len(tracks)) : 
			if i != j :
				a = tracks[i]
				b = tracks[j]
				G.add_edge(a, b, track_id=f"{a} - {b}", weight=tracks_dict[edges_ids[a]]["distance"])

In [30]:
def find_path_between_stations(G, station_A, station_B, station_A_tracks, station_B_tracks, station_to_station_paths):
	paths = None
	if (station_A, station_B) not in station_to_station_paths :
		station_to_station_paths[(station_A, station_B)] = dict()
		for platform_A in station_A_tracks :
			for platform_B in station_B_tracks :
				if (platform_A, platform_B) not in station_to_station_paths[(station_A, station_B)] :
					print(f"searching path between {platform_A} and {platform_B}")
					try:
						path = nx.shortest_path(G, platform_A, platform_B, weight="weight")
						if len(path) == 0 :
							path = None
						else :
							length = nx.path_weight(G, path, weight="weight")
						if path is not None :
							station_to_station_paths[(station_A, station_B)][(platform_A, platform_B)] = (path, length)
						else :
							station_to_station_paths[(station_A, station_B)][(platform_A, platform_B)] = None
					except:
						print("error")
						continue

In [ ]:
station_to_station_paths = dict()
for trip_id in final_trips :
	full_path = []
	path = final_trips[trip_id]

	for i in range(len(path) - 1):
		print(f"Finding path between {platforms_dict[path[i]]['name']} and {platforms_dict[path[i + 1]]['name']}")
		tracks_A = platforms_dict[path[i]]["platforms"]
		tracks_B = platforms_dict[path[i + 1]]["platforms"]

		find_path_between_stations(G, path[i], path[i + 1], tracks_A, tracks_B, station_to_station_paths)

Finding path between HALLE and FOREST-MIDI
searching path between 6591 and 4472
searching path between 6591 and 4352
searching path between 6591 and 4286
searching path between 6591 and 22107
searching path between 5601 and 4472
searching path between 5601 and 4352
searching path between 5601 and 4286
searching path between 5601 and 22107
searching path between 5598 and 4472
searching path between 5598 and 4352
searching path between 5598 and 4286
searching path between 5598 and 22107
searching path between 18289 and 4472
searching path between 18289 and 4352
searching path between 18289 and 4286
searching path between 18289 and 22107
searching path between 5555 and 4472
searching path between 5555 and 4352
searching path between 5555 and 4286
searching path between 5555 and 22107
Finding path between FOREST-MIDI and BRUXELLES-MIDI
searching path between 4472 and 4350
searching path between 4472 and 4883
searching path between 4472 and 4797
searching path between 4472 and 4804
searchin

In [27]:
sumo_home = os.environ.get("SUMO_HOME")
if not sumo_home:
	raise RuntimeError("Environment variable SUMO_HOME is not defined.")
tools = os.path.join(sumo_home, "tools")
tool_path = os.path.join(tools, "randomTrips.py")
if not os.path.isfile(tool_path):
	raise RuntimeError(f"randomTrips.py not found in {tools}")

cmd = [
	"py",								# ensure Python is available
	tool_path,		# path to the randomTrips.py script
	"-n", f"{OUTPUT_DIRECTORY}/network.net.xml",				# input: the compiled SUMO network
	"-o", f"{OUTPUT_DIRECTORY}/schedule.trips.xml",			# output: generated trips file
	"-a", f"sumo_data/trains.add.xml",				# use the defined vehicle type(s)
	"--trip-attributes", 'type="myTrain"',				# assign the vType 'myTrain' to each trip
	"--edge-permission", "rail",						# ensure only rail edges are used
	"-b", str(0),				# begin time for trip generation
	"-e", str(86400),			# end time for trip generation
	"--insertion-rate", "200,600,1100,700,500,900,1200,400",		# train insertion rate (s)	
	"--min-distance", "1000",		# minimum origin-destination distance
	"--weights-prefix", f"{OUTPUT_DIRECTORY}/weights"
]
print(" ".join(cmd))

py C:\Program Files (x86)\Eclipse\Sumo\tools\randomTrips.py -n new_start/network.net.xml -o new_start/schedule.trips.xml -a sumo_data/trains.add.xml --trip-attributes type="myTrain" --edge-permission rail -b 0 -e 86400 --insertion-rate 200,600,1100,700,500,900,1200,400 --min-distance 1000 --weights-prefix new_start/weights


In [28]:
start_time = time()
subprocess.run(cmd, check=True)
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")

Total execution time: 15.26 seconds.


In [29]:
import shutil

path = shutil.which("duarouter")

cmd = [
	f"{path}",						# ensure duarouter is available
	"--net-file", f"{OUTPUT_DIRECTORY}/network.net.xml",			# input: compiled network
	"--route-files", f"{OUTPUT_DIRECTORY}/schedule.trips.xml",		# input: generated trips file
	"--additional-files", f"sumo_data/trains.add.xml",			# input: additional file with vehicle types
	"--output-file", f"{OUTPUT_DIRECTORY}/routes.rou.xml",			# output: generated routes file
	"--ignore-errors", "true",						# stop if any routing error occurs
]
print(" ".join(cmd))

C:\Program Files (x86)\Eclipse\Sumo\bin\duarouter.EXE --net-file new_start/network.net.xml --route-files new_start/schedule.trips.xml --additional-files sumo_data/trains.add.xml --output-file new_start/routes.rou.xml --ignore-errors true


In [ ]:
start_time = time()
subprocess.run(cmd, check=True)
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")

## Simulation configuration

In [30]:
start_time = time()
sumo_config_str : str = (f'<?xml version="1.0" encoding="UTF-8"?>\n' +
'<configuration>\n' +
	'\t<input>\n' +
		'\t\t<net-file value="network.net.xml"/>\n' +
		'\t\t<route-files value="routes.rou.xml"/>\n' +
		# '\t\t<additional-files value="platforms.add.xml"/>\n' +
	'\t</input>\n' +
	'\t<time>\n' +
		'\t\t<begin value="0"/>\n' +
		f'\t\t<end value="{1 * 24 * 3600}"/>\n' +
	'\t</time>\n' +
	'\t<report>\n' +
		'\t\t<no-step-log value="true"/>\n' +
	'\t</report>\n' +
	'\t<output>\n' +
		'\t\t<tripinfo-output value="tripinfo.xml"/>\n' +
		'\t\t<stop-output value="stopinfo.xml"/>\n' +
		'\t\t<summary-output value="summary.xml"/>\n' +
	'\t</output>\n' +
'</configuration>'
)
end_time = time()
print(f"Total execution time: {end_time - start_time:.2f} seconds.")
# print(sumo_config_str)

Total execution time: 0.00 seconds.


In [ ]:
with open(output["config"], 'w', encoding = "utf-8") as f :
		f.write(sumo_config_str)